# Lecture 6 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week06/lecture06_exercise.ipynb`

**Rules:**
1. Use `px` first, then customise with `update_traces` / `update_layout`
2. Colour encodes a meaningful category — not decoration
3. Insight title names the specific finding
4. Consider: would a bar chart be clearer? If yes, use the bar chart

---


In [ ]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv('../data/global_energy_mix.csv')

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [ ]:
# Task 1 — Treemap: fossil fuel dependency by country

# Filter to fossil sources only
fossil_df = df.loc[df['Source_Type'] == 'Fossil'].copy()

# CVD-safe palette for the three fossil sources
fossil_color_map = {
    'Coal':        '#D55E00',   # vermillion
    'Oil':         '#E69F00',   # amber
    'Natural Gas': '#F0E442',   # yellow
}

# ── Step 1: Plotly Express base chart ─────────────────────────────────────────
fig1 = px.treemap(
    fossil_df,
    path=['Region', 'Country', 'Source'],
    values='TWh',
    color='Source',
    color_discrete_map=fossil_color_map,
    labels={'TWh': 'Energy (TWh)', 'Source': 'Fossil Source'},
    height=700, width=1300
)

# ── Step 2: Customisation ─────────────────────────────────────────────────────
fig1.update_traces(
    textinfo='label+value',
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    hovertemplate='<b>%{label}</b><br>%{value:,.0f} TWh<extra></extra>',
)

# Grey out parent nodes (Region and Country levels)
fig1.data[0].marker.colors = [
    c if c in fossil_color_map.values() else '#DDDDDD'
    for c in fig1.data[0].marker.colors
]

fig1.update_layout(
    title='Asia-Pacific dominates global fossil fuel consumption — coal is the leading source',
    font=dict(family='Arial', size=12),
    margin=dict(l=10, r=10, t=55, b=10),
    paper_bgcolor='white',
)

fig1.show()


## Task 2 — Sunburst: tipping behaviour by day and meal time

**What to build:** A sunburst chart using the built-in `tips` dataset showing how **total bill amount** is distributed across day → time → smoker status.

**Requirements:**
- Load tips with `px.data.tips()`
- Aggregate **total bill** (sum of `total_bill`) per group — not count
- Hierarchy: `path=['day', 'time', 'smoker']`
- Colour encodes smoker status with a CVD-safe blue/orange palette
- Grey out parent nodes (day and time level)
- Use `percent parent` for text labels
- Insight title describing where the most spending happens

> 💡 `tips.groupby(['day', 'time', 'smoker'])['total_bill'].sum().reset_index()`


In [ ]:
# Task 2 — Sunburst: tipping behaviour by day and meal time

tips = px.data.tips()

# Aggregate total bill per group
df_tips = (
    tips.groupby(['day', 'time', 'smoker'])['total_bill']
    .sum()
    .reset_index()
)

# CVD-safe palette: blue for No, orange for Yes (smoker)
smoker_color_map = {
    'No':  '#0072B2',   # blue
    'Yes': '#E69F00',   # orange
}

# ── Step 1: Plotly Express base chart ─────────────────────────────────────────
fig2 = px.sunburst(
    df_tips,
    path=['day', 'time', 'smoker'],
    values='total_bill',
    color='smoker',
    color_discrete_map=smoker_color_map,
    labels={'total_bill': 'Total Bill ($)', 'smoker': 'Smoker'},
    height=650, width=650
)

# ── Step 2: Customisation ─────────────────────────────────────────────────────
fig2.update_traces(
    textinfo='label+percent parent',
    hovertemplate='<b>%{label}</b><br>Total Bill: $%{value:,.2f}<br>%{percentParent:.0%} of %{parent}<extra></extra>',
    insidetextorientation='radial',
)

# Grey out parent nodes (day and time levels)
fig2.data[0].marker.colors = [
    c if c in smoker_color_map.values() else '#DDDDDD'
    for c in fig2.data[0].marker.colors
]

fig2.update_layout(
    title="Saturday dinner drives the most spending — non-smokers consistently outspend smokers",
    font=dict(family='Arial', size=12),
    margin=dict(l=10, r=10, t=55, b=10),
    paper_bgcolor='white',
)

fig2.show()


## Task 3 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country


In [ ]:
# Task 3 — Treemap vs bar: low-carbon energy by country

# Filter and aggregate
low_carbon = (
    df.loc[df['Source_Type'] == 'Low-carbon']
    .groupby('Country')['TWh']
    .sum()
    .reset_index()
    .sort_values('TWh')
)
low_carbon['All'] = 'Low-carbon'   # dummy root node

# ── Treemap ───────────────────────────────────────────────────────────────────
fig_tree = px.treemap(
    low_carbon,
    path=['All', 'Country'],
    values='TWh',
    color='TWh',
    color_continuous_scale='Blues',
    labels={'TWh': 'Energy (TWh)'},
    title='Low-carbon energy by country',
    height=500, width=1000,
)
fig_tree.update_traces(
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    hovertemplate='<b>%{label}</b><br>%{value:,.0f} TWh<extra></extra>',
    root_color='white',
)
fig_tree.update_layout(
    font=dict(family='Arial', size=12),
    margin=dict(l=10, r=10, t=55, b=10),
    paper_bgcolor='white',
)
fig_tree.show()

# ── Bar chart ─────────────────────────────────────────────────────────────────
fig_bar = px.bar(
    low_carbon,
    x='TWh',
    y='Country',
    orientation='h',
    color='TWh',
    color_continuous_scale='Blues',
    labels={'TWh': 'Low-carbon Energy (TWh)', 'Country': ''},
    title='China leads on low-carbon energy, driven by massive hydro capacity',
    height=500, width=900,
)
fig_bar.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    coloraxis_showscale=False,
    margin=dict(l=10, r=10, t=55, b=10),
)
fig_bar.update_xaxes(gridcolor='#EEEEEE')
fig_bar.update_yaxes(gridcolor='#EEEEEE')
fig_bar.show()
